In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable
from datetime import datetime

start_time = datetime.now()

### Audit

In [0]:
def audit_log(run_id, schema_name, table_name, start_time, status, row_count, error):
    end_time = datetime.now()
    log = {
        'run_id': run_id,
        'schema_name': schema_name,
        'table_name': table_name,
        'start_time': str(start_time),
        'end_time': str(end_time),
        'duration': (end_time - start_time).total_seconds(),
        'status': status,
        'rows_written': int(row_count or 0),
        'error_message': error
    }
    schema = ("run_id string, schema_name string, table_name string, "
              "start_time string, end_time string, duration double, "
              "status string, rows_written bigint, error_message string")

    spark.createDataFrame([log], schema) \
         .write.mode('append').saveAsTable('retail.ops.audit_logs')

def update_watermark(schema, table, latest_watermark):
    if latest_watermark is None:
        return
    spark.sql(f"""
        UPDATE retail.ops.watermark
        SET last_watermark = TIMESTAMP '{latest_watermark}'
        WHERE schema_name = '{schema}' AND table_name = '{table}'
    """)

### Load Task Parameters

In [0]:
# dbutils.widgets.text('catalog', 'retail', 'Catalog')
# dbutils.widgets.text('schema', 'silver', 'Schema')
# dbutils.widgets.text('table', 'customers', 'Table')
# dbutils.widgets.text('run_id', 'test', 'Run ID')
# dbutils.widgets.text('watermark_column', '_ingested_at', 'Watermark Column')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table = dbutils.widgets.get('table')
run_id = dbutils.widgets.get('run_id')
watermark_column = dbutils.widgets.get('watermark_column')

### Latest Watermark

In [0]:
latest_watermark= spark.sql(f'''
          SELECT MAX(last_watermark) FROM {catalog}.ops.watermark
          WHERE schema_name = '{schema}' AND table_name = '{table}'
          ''').first()[0]

In [0]:
latest_watermark

### Read Bronze Table Incrementally

In [0]:
bronze_batch = (
    spark.read.table(f'{catalog}.bronze.{table}') 
    .filter(F.col('_ingested_at') > latest_watermark)
)

silver_watermark = bronze_batch.select(F.max('_ingested_at')).first()[0]

### Transformation

In [0]:
## transformation
customer_df = (
    bronze_batch    
    .withColumn('customer_id', F.trim('customer_id'))
    .withColumn('first_name', F.initcap(F.trim('first_name')))
    .withColumn('last_name',  F.initcap(F.trim('last_name')))
    .withColumn('email', F.lower(F.trim('email')))
    .withColumn('city', F.initcap(F.trim('city')))
    .withColumn('segment', F.initcap(F.trim('segment')))
    .withColumn('updated_at', F.to_timestamp(F.col('updated_at'), 'yyyy-MM-dd HH:mm:ss'))
    .withColumnRenamed('_ingested_at', '_bronze_ingested_at')
    .withColumnRenamed('run_id','_bronze_run_id')
    .withColumn('_silver_updated_at', F.current_timestamp())   
)

## Dedupe over customer_ids
w = Window.partitionBy('customer_id').orderBy(F.col('updated_at').desc(), F.col('file_modification_time').desc())
customer_df = (
    customer_df
    .withColumn('rn', F.row_number().over(w))
    .filter(F.col('rn')==1)
    .drop('rn', 'file_modification_time')
)

### Write the table to Silver Schema

- SCD Type 1
- SCD Type 2

### DQ Checks

In [0]:
dq_rules = {
    'customer_id_missing': "customer_id IS NULL OR customer_id = ''",
    'updated_at_invalid':  "updated_at IS NULL", 
    'email_invalid':       r"email IS NOT NULL AND NOT email RLIKE '^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$'",
    'phone_invalid':       r"phone IS NOT NULL AND NOT phone RLIKE '^[0-9]{10,11}$'"
}

if not customer_df.isEmpty():
    dq_checks = (
        customer_df.withColumn('dq_checks', F.array_compact(F.array(*[F.when(F.expr(cond), F.lit(name)) for name, cond in dq_rules.items()])))
    )

    valid_df = dq_checks.filter(F.size('dq_checks')==0).drop('dq_checks')

    invalid_df = (
        dq_checks
        .filter(F.size('dq_checks')>0)
        .withColumn('_dq_run_id', F.lit(run_id))
        .withColumn('_quarantied_at', F.current_timestamp())
    )

    if not invalid_df.isEmpty():
        (
            invalid_df
            .write
            .mode('append')
            .option('mergeSchema', 'true')
            .saveAsTable(f'{catalog}.{schema}.dq_{table}')
        )

In [0]:
display(valid_df)

## Write the data (SCD TYPE 2 Logic)

In [0]:
STATUS = None
error = None
rows_written = 0

try:
    if customer_df.isEmpty():
        STATUS = 'SKIPPED'
    else:
        
        target_table_name = f'{catalog}.{schema}.{table}'

        scd_columns = ['first_name', 'last_name', 'email', 'phone', 'city', 'segment']
        src = (
            valid_df
            .withColumn(
                'row_hash', 
                F.sha2(F.concat_ws("||", *scd_columns), 256)
            )
        )
        
        if not spark.catalog.tableExists(target_table_name):
            (
                src
                .withColumn('is_current', F.lit(True))
                .withColumn('effective_from', F.current_timestamp())
                .withColumn('effective_to', F.lit(None).cast('TIMESTAMP'))
                .write
                .mode('append')
                .saveAsTable(target_table_name)
            )
            rows_written = src.count()
            
        else:

            target = DeltaTable.forName(spark, target_table_name)
            ## First deactivate the records already existing
            (
                target.alias('t')
                .merge(
                    src.alias('s'),
                    't.customer_id == s.customer_id'
                )
                .whenMatchedUpdate(
                    condition = 't.row_hash <> s.row_hash AND s.updated_at > t.updated_at',
                    set = {
                        'is_current': 'false',
                        'effective_to': F.current_timestamp()
                    }
                )
                .execute()
            )

            ## Insert all the current records that have actually changed, for those with no changes at all -> skip it 

            current_df = target.toDF().filter(F.col('is_current')==True)

            ## Only filter out the new rows
            to_insert = (
                src.alias('s')
                .join(
                    current_df.alias('c'),
                    F.col('s.customer_id') == F.col('c.customer_id'),
                    'left_anti'
                )
                .withColumn('is_current', F.lit(True))
                .withColumn('effective_from', F.current_timestamp())
                .withColumn('effective_to', F.lit(None).cast('TIMESTAMP'))
            )

            (
                to_insert
                .write
                .mode('append')
                .saveAsTable(target_table_name)
            )

            rows_written = to_insert.count()

        STATUS = "SUCCESS"

        if silver_watermark is not None:
            update_watermark(schema, table, silver_watermark)

except Exception as e:
    error = str(e)[:1000]
    STATUS = "FAILURE"
    raise

finally: 
    audit_log(
        run_id=run_id,
        schema_name=schema,
        table_name=table,
        start_time=start_time,
        status=STATUS,
        row_count=rows_written,
        error=error
    )